# Bayesian Analysis 03 — Bayesian Time Series Model

**Goal:** Build a Bayesian price prediction model that produces full posterior credible intervals for t+7 and t+30. Instead of a point forecast the model says: 'Card XY will cost €2.50 [€1.80, €3.40] with 90% credibility'.

**Tables:** gold_price_features (full history per card)

**Model:** Local Level Model (Bayesian equivalent of a random walk with noise)
- `μ_t = μ_{t-1} + η_t` — trend evolution
- `log1p(EUR_t) = μ_t + ε_t` — observed price
- Simple, interpretable, appropriate for short price histories

**Why Local Level and not SARIMA:** With only ~90 days of data, SARIMA with many parameters overfits. The Local Level model has just 2 parameters (σ_η and σ_ε) — the minimal model capable of capturing price drift.

⚠️ **Data requirement:** Minimum 30 days of history per card. Cards with <30 days → skip; use the prior from the hierarchical model (BA-02) instead.

In [1]:
import duckdb
import numpy as np
import pandas as pd

In [2]:
silver = duckdb.connect("../../data/silver/cards.duckdb", read_only=True)
gold = duckdb.connect("../../data/gold/cards.duckdb", read_only=True)

In [3]:
df = gold.execute("""
    SELECT p.uuid, p.snapshot_date, p.eur
    FROM gold_price_features p
    WHERE p.eur IS NOT NULL AND p.uuid IS NOT NULL
    ORDER BY p.uuid, p.snapshot_date
""").df()
df["snapshot_date"] = pd.to_datetime(df["snapshot_date"])
df["log_eur"] = np.log1p(df["eur"])

names_df = silver.execute(
    "SELECT uuid, name FROM silver_cards WHERE uuid IS NOT NULL"
).df()

n_snapshots = df["snapshot_date"].nunique()
MIN_OBS = 30

print(f"Total cards:    {df['uuid'].nunique():,}")
print(f"Snapshots:      {n_snapshots}  (need >= {MIN_OBS} for Local Level Model)")

if n_snapshots < MIN_OBS:
    rerun = (
        df["snapshot_date"].max() + pd.Timedelta(days=MIN_OBS - n_snapshots)
    ).date()
    print(f"\n  DEFERRED: only {n_snapshots} snapshots available.")
    print(f"  Re-run after approximately {rerun} (need >= {MIN_OBS} daily snapshots).")
    print()
    print("  When re-run, this notebook will:")
    print("   1. Select 5 representative cards with >= 30 days history")
    print("   2. Fit Local Level Model per card (GaussianRandomWalk in PyMC)")
    print("   3. Forecast t+7 and t+30 with 50%/90% HDI ribbons")
    print("   4. Compare MAE vs naive baseline")
    print("   5. Report HDI calibration (expect ~90% coverage)")
else:
    # Identify cards with sufficient history
    obs_per_card = df.groupby("uuid")["snapshot_date"].nunique()
    eligible = obs_per_card[obs_per_card >= MIN_OBS].index
    print(f"  Cards with >= {MIN_OBS} snapshots: {len(eligible):,}")
    print("  Proceed to Section 1 to build models.")

Total cards:    82,876
Snapshots:      36  (need >= 30 for Local Level Model)


  Cards with >= 30 snapshots: 80,750
  Proceed to Section 1 to build models.


In [4]:
TEST_SIZE = 7

if n_snapshots < MIN_OBS:
    print("Card selection deferred - insufficient snapshots.")
else:
    eligible_df = df[df["uuid"].isin(eligible)]
    per_card_stats = (
        eligible_df.groupby("uuid")
        .agg(
            last_eur=("eur", "last"),
            std_log_eur=("log_eur", "std"),
        )
        .reset_index()
    )

    def _assign_tier(eur):
        if eur >= 1000:
            return 3
        if eur >= 100:
            return 2
        return 1

    per_card_stats["tier"] = per_card_stats["last_eur"].apply(_assign_tier)

    # Representative selection: Tier 1 stable, Tier 1 spike, Tier 2, Tier 3, highest-volatility overall.
    picks = {}
    tier1 = per_card_stats[per_card_stats["tier"] == 1].sort_values("std_log_eur")
    if len(tier1) > 0:
        picks["tier1_stable"] = tier1.iloc[0]["uuid"]
    if len(tier1) > 1:
        picks["tier1_spike"] = tier1.iloc[-1]["uuid"]
    tier2 = per_card_stats[per_card_stats["tier"] == 2].sort_values("std_log_eur")
    if len(tier2) > 0:
        picks["tier2"] = tier2.iloc[-1]["uuid"]
    tier3 = per_card_stats[per_card_stats["tier"] == 3].sort_values("std_log_eur")
    if len(tier3) > 0:
        picks["tier3"] = tier3.iloc[-1]["uuid"]
    if len(per_card_stats) > 0:
        picks["high_vol"] = per_card_stats.sort_values("std_log_eur").iloc[-1]["uuid"]

    selected_uuids = list(dict.fromkeys(picks.values()))
    name_lookup = names_df.set_index("uuid")["name"].to_dict()

    selected_cards = {}
    for uuid_ in selected_uuids:
        series = (
            eligible_df[eligible_df["uuid"] == uuid_]
            .sort_values("snapshot_date")["log_eur"]
            .to_numpy()
        )
        label = name_lookup.get(uuid_, uuid_)
        selected_cards[label] = {
            "log1p_eur_train": series[:-TEST_SIZE],
            "log1p_eur_test": series[-TEST_SIZE:],
        }

    print(f"Selected {len(selected_cards)} representative cards:")
    for label, data in selected_cards.items():
        print(
            f"  {label}: train={len(data['log1p_eur_train'])} obs,"
            f" test={len(data['log1p_eur_test'])} obs"
        )

Selected 4 representative cards:
  Plumes of Peace: train=29 obs, test=7 obs
  Horned Turtle: train=29 obs, test=7 obs
  The Master, Mesmerist: train=29 obs, test=7 obs
  Tinker: train=29 obs, test=7 obs


## 1. Local Level Model — Specification

**Mathematical model:**
```
μ_1 ~ Normal(log1p(eur_1), sigma_eta)   # initialised from the first observed price

η_t ~ Normal(0, sigma_eta)               # trend innovations
μ_t = μ_{t-1} + η_t  for t=2,...,T       # trend evolution

ε_t ~ Normal(0, sigma_eps)               # observation noise
log1p(EUR_t) = μ_t + ε_t                 # observed price
```

**Parameter interpretation:**
- `sigma_eta` (small) → stable prices, slow drift
- `sigma_eta` (large) → fast-moving prices, speculative
- `sigma_eps` (small) → prices are observed with little noise
- `sigma_eps` (large) → data is noisy

**Forecasting:** After training, sample μ_{T+7} and μ_{T+30} from the posterior.

In [5]:
def build_local_level_model(y_train):
    """Local Level Model: mu_t = mu_{t-1} + eta_t, y_t = mu_t + eps_t."""
    import pymc as pm

    T = len(y_train)
    with pm.Model() as model:
        sigma_eta = pm.HalfNormal("sigma_eta", sigma=0.1)
        sigma_eps = pm.HalfNormal("sigma_eps", sigma=0.1)
        mu_init = pm.Normal("mu_init", mu=float(y_train[0]), sigma=0.5)
        mu = pm.GaussianRandomWalk(
            "mu", sigma=sigma_eta, init_dist=pm.Normal.dist(mu_init, 0.1), shape=T
        )
        _ = pm.Normal("y_obs", mu=mu, sigma=sigma_eps, observed=y_train)
    return model


if n_snapshots < MIN_OBS:
    rerun = (
        df["snapshot_date"].max() + pd.Timedelta(days=MIN_OBS - n_snapshots)
    ).date()
    print(f"DEFERRED: {n_snapshots} snapshots (need >= {MIN_OBS}).")
    print(f"  Re-run after approximately {rerun}.")
    print()
    print("  Model blueprint defined above — will be applied to 5 cards:")
    print("   - Tier 1 stable, Tier 1 spike, Tier 2, Tier 3, high-volatility")
else:
    # Build and verify model on first eligible card
    card_uuid = eligible[0]
    y_train_card = (
        df[df["uuid"] == card_uuid].sort_values("snapshot_date")["log_eur"].values
    )
    model_test = build_local_level_model(y_train_card)
    print("Model built successfully")
    print(f"  Card: {card_uuid}  |  T={len(y_train_card)} observations")

g++ not available, if using conda: `conda install gxx`


Model built successfully
  Card: 0001e0d0-2dcd-5640-aadc-a84765cf5fc9  |  T=36 observations


## 2. Sampling and Diagnostics

**Separate model per card** — each card has its own σ_eta and σ_eps (different price dynamics).

**Sampling settings:**
- draws=2000, tune=1000, chains=2 (faster than 4 — this is a simple model)
- target_accept=0.9

**Mandatory diagnostics (same as always):**
- R-hat < 1.01
- ESS > 400
- 0 divergences

In [6]:
if n_snapshots < MIN_OBS:
    rerun = (
        df["snapshot_date"].max() + pd.Timedelta(days=MIN_OBS - n_snapshots)
    ).date()
    print(f"DEFERRED: {n_snapshots} snapshots (need >= {MIN_OBS}).")
    print(f"  Re-run after approximately {rerun}.")
    print()
    print("  Sampling will use: draws=2000, tune=1000, chains=2, target_accept=0.9")
    print("  Mandatory diagnostics per card:")
    print("    R-hat < 1.01, ESS > 400, divergences = 0")
else:
    import pymc as pm
    import arviz as az

    traces = {}
    for card_name, card_data in selected_cards.items():  # noqa: F821
        y_train = card_data["log1p_eur_train"]
        model = build_local_level_model(y_train)
        with model:
            trace = pm.sample(
                draws=2000,
                tune=1000,
                chains=2,
                target_accept=0.9,
                random_seed=42,
                return_inferencedata=True,
                progressbar=False,
            )
        summary = az.summary(trace, var_names=["sigma_eta", "sigma_eps"])
        rhat_ok = (pd.to_numeric(summary["r_hat"]) < 1.01).all()
        ess_ok = (summary["ess_bulk"] > 400).all()
        n_diverg = int(trace.sample_stats.diverging.sum())
        print(
            f"{card_name}: R-hat OK={rhat_ok}, ESS OK={ess_ok}, divergences={n_diverg}"
        )
        traces[card_name] = trace

Initializing NUTS using jitter+adapt_diag...


Multiprocess sampling (2 chains in 2 jobs)


NUTS: [sigma_eta, sigma_eps, mu_init, mu]


Sampling 2 chains for 1_000 tune and 2_000 draw iterations (2_000 + 4_000 draws total) took 43 seconds.


Chain 0 reached the maximum tree depth. Increase `max_treedepth`, increase `target_accept` or reparameterize.


...

Chain 1 reached the maximum tree depth. Increase `max_treedepth`, increase `target_accept` or reparameterize.


We recommend running at least 4 chains for robust computation of convergence diagnostics


The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details


The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details


Tinker: R-hat OK=False, ESS OK=False, divergences=0


## 3. t+7 and t+30 Forecast with Posterior Credible Intervals

**Method:** Extend the model by T+7 and T+30 future steps. Sample μ_{T+7} and μ_{T+30} from the posterior.

**Output:** Not a point prediction but a posterior distribution over future prices. The 90% Highest Density Interval (HDI) = the price range that the model considers 90% probable.

**Visualisation:** Line chart with:
- Dark line = observed prices (train + test)
- Coloured ribbons = 50% and 90% HDI for the forecast
- Vertical dashed line = train/test split point

**Evaluation:** Do the observed test-period prices fall within the 90% HDI? (expect ~90% of test points to be covered)

In [7]:
def predict_future(trace, n_future=30):
    """Sample future prices from the posterior of the Local Level Model."""
    mu_last = trace.posterior["mu"].values[:, :, -1]
    sigma_eta = trace.posterior["sigma_eta"].values
    sigma_eps = trace.posterior["sigma_eps"].values

    n_samples = mu_last.shape[0] * mu_last.shape[1]
    mu_flat = mu_last.reshape(-1)
    se_flat = sigma_eta.reshape(-1)
    sp_flat = sigma_eps.reshape(-1)

    rng_f = np.random.default_rng(99)
    predictions = np.zeros((n_samples, n_future))
    mu_cur = mu_flat.copy()
    for t in range(n_future):
        mu_cur = mu_cur + rng_f.normal(0, se_flat)
        predictions[:, t] = mu_cur + rng_f.normal(0, sp_flat)
    return predictions


if n_snapshots < MIN_OBS:
    rerun = (
        df["snapshot_date"].max() + pd.Timedelta(days=MIN_OBS - n_snapshots)
    ).date()
    print(f"DEFERRED: {n_snapshots} snapshots (need >= {MIN_OBS}).")
    print(f"  Re-run after approximately {rerun}.")
    print()
    print("  predict_future() defined above — will produce (n_samples, 30) array")
    print(
        "  Plots: observed + 50%/90% HDI ribbons, vertical split at train/test boundary"
    )
else:
    import arviz as az

    for card_name, trace in traces.items():
        preds = predict_future(trace, n_future=30)
        # plot ribbon chart for each card
        print(f"{card_name}: median t+7 = {np.expm1(np.median(preds[:, 6])):.3f} EUR")

Plumes of Peace: median t+7 = 0.070 EUR
Horned Turtle: median t+7 = 0.130 EUR
The Master, Mesmerist: median t+7 = 0.670 EUR
Tinker: median t+7 = 1260.000 EUR


## 4. Comparison Against a Naive Baseline

**Method:** Compare median Bayesian forecasts against naive forecast (last observed price = prediction) and 7-day moving average on the test set.

**Metrics:** MAE and RMSE per card. The Bayesian model wins if MAE < MAE_naive.

**Key interpretation:** The advantage of the Bayesian model is NOT point forecast accuracy (it may be similar to naive) — it is **interval calibration**. A model that says '90% HDI [€1.80, €3.40]' and covers 90% of observations is better than a model that says '€2.50' with no uncertainty quantification.

In [8]:
if n_snapshots < MIN_OBS:
    rerun = (
        df["snapshot_date"].max() + pd.Timedelta(days=MIN_OBS - n_snapshots)
    ).date()
    print(f"DEFERRED: {n_snapshots} snapshots (need >= {MIN_OBS}).")
    print(f"  Re-run after approximately {rerun}.")
    print()
    print("  Baseline comparison will compute per card:")
    print("    MAE_bayes  = |median_pred_7d - y_test_7|")
    print("    MAE_naive  = |y_train_last - y_test_7|  (last-observed forecast)")
    print("    HDI_cover  = fraction of test points inside 90% HDI")
    print()
    print("  Primary advantage is HDI calibration, not point accuracy.")
else:
    import arviz as az

    results = []
    for card_name, trace in traces.items():  # noqa: F821
        y_test = selected_cards[card_name]["log1p_eur_test"]  # noqa: F821
        y_train = selected_cards[card_name]["log1p_eur_train"]  # noqa: F821
        preds = predict_future(trace, n_future=max(30, len(y_test)))
        n_test = min(7, len(y_test))
        mae_bayes = abs(
            np.median(preds[:, min(6, len(y_test) - 1)])
            - y_test[min(6, len(y_test) - 1)]
        )
        mae_naive = abs(y_train[-1] - y_test[min(6, len(y_test) - 1)])
        hdi_90 = az.hdi(preds[:, :n_test], prob=0.90)
        coverage = np.mean(
            [
                (y_test[t] >= hdi_90[t, 0]) & (y_test[t] <= hdi_90[t, 1])
                for t in range(n_test)
            ]
        )
        results.append(
            {
                "card": card_name,
                "MAE_bayes": round(mae_bayes, 4),
                "MAE_naive": round(mae_naive, 4),
                "HDI_cover": round(coverage, 2),
            }
        )
    print(pd.DataFrame(results).to_string(index=False))

                 card  MAE_bayes  MAE_naive  HDI_cover
      Plumes of Peace     0.0000     0.0000       0.86
        Horned Turtle     4.4327     4.4327       0.57
The Master, Mesmerist     4.7077     4.7077       0.43
               Tinker     1.2112     1.2112       0.43


In [9]:
silver.close()
gold.close()

## 📋 Final Conclusions

```
STATUS: EXECUTED (data gate met — no longer deferred)
─────────────────────────────────────────────
Available snapshots: 36  (>= 30 required)
Total cards with any price history: 82,876
Cards with >= 30 snapshots (eligible): 80,750

CARD SELECTION
─────────────────────────────────────────────
Selected 4 representative cards (tier-based picks collapsed to 4 unique cards):
  Plumes of Peace          train=29 obs, test=7 obs
  Horned Turtle            train=29 obs, test=7 obs
  The Master, Mesmerist    train=29 obs, test=7 obs
  Tinker                   train=29 obs, test=7 obs

MODEL
─────────────────────────────────────────────
Model type: Local Level (Bayesian random walk with observation noise)
Sampler: NUTS, 2000 draws, 1000 tune, 2 chains, target_accept=0.9

MCMC DIAGNOSTICS — FAILED for all 4 cards
─────────────────────────────────────────────
Plumes of Peace:       R-hat OK=False, ESS OK=False, divergences=0
Horned Turtle:         R-hat OK=False, ESS OK=False, divergences=0
The Master, Mesmerist: R-hat OK=False, ESS OK=False, divergences=0
Tinker:                R-hat OK=False, ESS OK=False, divergences=0
Every chain hit the maximum tree depth; PyMC flagged R-hat > 1.01 and ESS per
chain < 100 for at least one parameter on every card, and recommended running
at least 4 chains.
→ None of the 4 per-card posteriors meet the mandatory convergence bar
  (R-hat < 1.01, ESS > 400, 0 divergences) — only the divergence count (0)
  passed for every card.
→ The forecasts, MAE, and HDI coverage below should be treated as provisional:
  sampler geometry was not adequately explored with only 2 chains / 2000 draws
  on these short (T=29) series. Re-run with more chains/draws, a higher
  target_accept or max_treedepth, and/or a non-centered reparameterization of
  the GaussianRandomWalk before trusting these numbers.

FORECASTS (t+7 median, posterior)
─────────────────────────────────────────────
Plumes of Peace:       €0.070
Horned Turtle:         €0.130
The Master, Mesmerist: €0.670
Tinker:                €1260.000

BASELINE COMPARISON (t+7)
─────────────────────────────────────────────
                 card  MAE_bayes  MAE_naive  HDI_cover
      Plumes of Peace     0.0000     0.0000       0.86
        Horned Turtle     4.4327     4.4327       0.57
The Master, Mesmerist     4.7077     4.7077       0.43
               Tinker     1.2112     1.2112       0.43

→ MAE_bayes == MAE_naive for 3 of 4 cards: with sigma_eta this small over a
  T=29 window, the posterior median forecast collapses onto the last training
  value, so the Bayesian point forecast offers no accuracy edge over the naive
  baseline here — consistent with the notebook's stated expectation that the
  advantage is calibration, not point accuracy.
→ HDI coverage (target ~90%) was only close to target for Plumes of Peace
  (86%); Horned Turtle, The Master Mesmerist, and Tinker all under-covered
  (57%, 43%, 43%) — the 90% credible interval is too narrow for 3 of 4 cards,
  consistent with the failed convergence diagnostics above (an unreliable
  posterior yields unreliable HDIs).

KEY TAKEAWAY
─────────────────────────────────────────────
The data gate (>= 30 snapshots) is now met and the pipeline runs end-to-end,
but sampling diagnostics failed for every representative card under the
current settings (2 chains, target_accept=0.9, T=29). The interval-calibration
promise of the Local Level Model — its stated primary value over a naive
forecast — is NOT yet demonstrated: HDI coverage was near the 90% target for
only 1 of 4 cards. Before using this model's intervals in production, address
the sampler warnings (more chains, higher target_accept/max_treedepth, or a
non-centered parameterization) and re-evaluate calibration.
```
